# Retrieval Augmented Generation (RAG)

---

## The Problem

Large documents (e.g. an 800-page financial report) cannot always fit into a single prompt.
You need a way to get relevant information to Claude without including everything.

---

## Option 1 — Stuff Everything into the Prompt
```python
prompt = f"""
Answer the user's question about the financial document.

<user_question>
{user_question}
</user_question>

<financial_document>
{financial_document}
</financial_document>
"""
```

**Limitations:**

| Problem | Detail |
|---|---|
| Hard token limit | Document may simply be too long |
| Reduced quality | Claude becomes less effective with very long prompts |
| Higher cost | More tokens = more expensive |
| Slower | Larger prompts take longer to process |

---

## Option 2 — RAG (Retrieval Augmented Generation)

Break the document into chunks **once** during preprocessing.
At query time, find only the most relevant chunks and include those.
```
Preprocessing (done once):
Document → split into chunks → store chunks

At query time:
User question → search chunks → find relevant ones → include in prompt
```

**Example:**
```
Question: "What risks does this company face?"
         |
         v
Search chunks → find "Risk Factors" section
         |
         v
Include only that chunk in the prompt → send to Claude
```

---

## RAG vs Prompt Stuffing

| | Prompt Stuffing | RAG |
|---|---|---|
| **Simplicity** | Simple | More complex |
| **Scale** | Limited by token window | Scales to any document size |
| **Cost** | Higher (full doc every time) | Lower (only relevant chunks) |
| **Speed** | Slower | Faster |
| **Multiple docs** | Not practical | Works well |
| **Context accuracy** | Full context always present | May miss surrounding context |

---

## Challenges with RAG

- Requires a **preprocessing step** to chunk documents
- Needs a **search mechanism** to find relevant chunks
- Included chunks may **lack surrounding context** Claude needs
- Many chunking strategies — no single best approach

### Chunking strategies (trade-offs to evaluate):
- Equal-sized portions (fixed token/character length)
- Structure-based splits (headers, sections, paragraphs)
- Semantic splits (by topic or meaning)

---

## When to Use RAG

Use RAG when:
- Documents are **too large** to fit in a single prompt
- Working with **multiple documents**
- You need to **optimise for cost and performance**
- Query patterns are focused (users ask about specific sections)

Stick with prompt stuffing when:
- Documents are small enough to fit comfortably
- Simplicity matters more than scalability
- Every part of the document is potentially relevant

---

## Key Takeaway

> RAG trades **simplicity for scalability**.  
> It requires more upfront work but enables you to work with document collections
> that would be impossible to handle by stuffing everything into a prompt.

# Text Chunking Strategies for RAG

---

## Why Chunking Matters

Poor chunking = wrong context inserted into prompts = wrong answers.

**Example:** A user asks "How many bugs did engineers fix this year?"
If chunks are split poorly, the word "bug" in a medical research section could be
returned instead of the software engineering section.

---

## The Four Strategies

### 1. Size-Based Chunking

Split text into equal-length character chunks.
```python
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunk_text = text[start_idx:end_idx]
        chunks.append(chunk_text)

        start_idx = (
            end_idx - chunk_overlap if end_idx < len(text) else len(text)
        )

    return chunks
```

- `chunk_overlap` ensures words/sentences aren't cut off at boundaries
- **Pros:** Simple, works with any document type including code
- **Cons:** May split mid-sentence, loses natural context

---

### 2. Structure-Based Chunking

Split on document structure — headers, sections, paragraphs.
```python
def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)
```

- **Pros:** Clean, meaningful chunks — each represents a complete section
- **Cons:** Only works if document structure is guaranteed (e.g. Markdown, HTML)
- Not reliable for plain text or unstructured PDFs

---

### 3. Sentence-Based Chunking

Split into sentences, then group into chunks with overlap.
```python
def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)

    chunks = []
    start_idx = 0

    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        current_chunk = sentences[start_idx:end_idx]
        chunks.append(" ".join(current_chunk))

        start_idx += max_sentences_per_chunk - overlap_sentences
        if start_idx < 0:
            start_idx = 0

    return chunks
```

- **Pros:** Respects sentence boundaries, good general-purpose approach
- **Cons:** Sentences still lack broader section context

---

### 4. Semantic-Based Chunking

Group sentences by meaning using NLP — sentences with related meaning stay together.

- **Pros:** Highest quality, most relevant chunks
- **Cons:** Computationally expensive, complex to implement
- Not practical for simple pipelines

---

## Comparison

| Strategy | Quality | Complexity | Works on any doc? |
|---|---|---|---|
| Size-based | Moderate | Low | Yes |
| Structure-based | High | Low | No — needs structured docs |
| Sentence-based | Good | Medium | Yes |
| Semantic-based | Highest | High | Yes |

---

## How to Choose

| Situation | Recommended strategy |
|---|---|
| You control document formatting | Structure-based |
| General text documents | Sentence-based |
| Mixed content types / code / fallback | Size-based with overlap |
| Maximum quality, cost is acceptable | Semantic-based |

> Size-based chunking with overlap is the most common **production default** —
> simple, reliable, works with anything.

---

## Key Takeaway

> There is no single best chunking strategy.  
> The right choice depends on your documents, use case, and acceptable complexity.  
> When in doubt, start with size-based + overlap and measure quality before optimising.

# Text Embeddings for Semantic Search

---

## The Search Problem in RAG

After chunking, you need to find which chunks are **most relevant** to the user's question.
Two approaches:

| Approach | How it works |
|---|---|
| **Keyword search** | Looks for exact word matches |
| **Semantic search** | Understands meaning and context — finds relevant chunks even without exact word matches |

Semantic search is the standard approach in RAG pipelines.

---

## What is a Text Embedding?

A **numerical representation of the meaning** contained in a piece of text.
```
Text → Embedding Model → [0.23, -0.87, 0.41, 0.05, ...]
                          (a long list of numbers, each between -1 and +1)
```

- Each number represents some **learned quality or feature** of the text
- The model learns what each dimension means during training
- We cannot directly interpret what each individual number means
- Conceptually: one number might loosely relate to "sentiment", another to "topic" —
  but these are not explicitly defined

---

## Why Embeddings Enable Semantic Search

Two pieces of text with **similar meaning** will produce embeddings with
**similar numerical values** — even if they use completely different words.
```
"What are the risk factors?"
"What dangers does the company face?"
→ These will produce similar embeddings → both match a "Risk Factors" chunk
```

---

## VoyageAI — Recommended Embedding Provider

Anthropic does not currently provide embedding generation.
Recommended provider: **VoyageAI** (free to start)

Setup:
```bash
%pip install voyageai
```

Add to `.env`:
```
VOYAGE_API_KEY="your_key_here"
```

---

## Implementation
```python
from dotenv import load_dotenv
import voyageai

load_dotenv()
client = voyageai.Client()

def generate_embedding(text, model="voyage-3-large", input_type="query"):
    result = client.embed([text], model=model, input_type=input_type)
    return result.embeddings[0]    # returns a list of floats
```

### Parameters

| Parameter | Purpose |
|---|---|
| `model` | Which VoyageAI model to use — `voyage-3-large` is the recommended default |
| `input_type` | `"query"` for user questions, `"document"` for chunks being indexed |

---

## RAG Pipeline Position
```
Document
    |
    v
Chunking
    |
    v
Generate embedding for each chunk → store
    |
    v
User asks a question
    |
    v
Generate embedding for the question
    |
    v
Compare question embedding to chunk embeddings → find closest matches
    |
    v
Include top matching chunks in prompt → send to Claude
```

---

## Key Takeaway

> Embeddings convert text meaning into numbers so that similarity can be
> **measured mathematically**.  
> The next step is learning how to compare embeddings — this is the core of
> semantic search in a RAG pipeline.

# The Full RAG Pipeline — Step by Step

---

## Overview
```
Source Document
      |
      v
Step 1: Chunk text
      |
      v
Step 2: Generate embeddings for each chunk
      |
      v
Step 3: Store in vector database
      |
      v      ← preprocessing ends here, wait for user query
Step 4: Embed the user's query
      |
      v
Step 5: Find most similar chunk embeddings (cosine similarity)
      |
      v
Step 6: Build prompt with relevant chunks → send to Claude
```

---

## Step 1 — Chunk the Source Document

Break document into manageable pieces:
```
Chunk 1 (Medical):   "This year saw significant strides in our understanding
                      of XDR-47, a 'bug' we have not seen before."

Chunk 2 (Software):  "This division dedicated significant effort to studying
                      various infection vectors in our distributed systems."
```

---

## Step 2 — Generate Embeddings

Each chunk is converted to a vector of numbers by an embedding model.

Conceptually (simplified 2D example):

| Chunk | Embedding | Meaning |
|---|---|---|
| Medical | `[0.944, 0.331]` | High medical, some software (due to "bug") |
| Software | `[0.295, 0.955]` | High software, some medical (due to "infection") |

The API **normalises** each vector to magnitude 1.0 automatically.

---

## Step 3 — Store in a Vector Database

A vector database is optimised for storing and searching through numerical embeddings.

Once stored, preprocessing is complete. The system waits for a user query.

---

## Step 4 — Embed the User Query
```
User: "What did the software engineering department do this year?"
         |
         v
Embedding: [0.112, 0.993]   ← low medical, high software score
```

The query is embedded using the **same model** used for chunks.

> Use `input_type="query"` for questions, `input_type="document"` for chunks.

---

## Step 5 — Find Similar Embeddings (Cosine Similarity)

The vector database compares the query embedding against all stored chunk embeddings.

### How Cosine Similarity Works

Measures the **cosine of the angle** between two vectors.

| Value | Meaning |
|---|---|
| Close to `1` | Very similar |
| `0` | No relationship |
| Close to `-1` | Very different |

**In our example:**

| Chunk | Cosine Similarity | Result |
|---|---|---|
| Software Engineering | 0.983 | High match — returned |
| Medical Research | 0.398 | Low match — not returned |

### Cosine Distance

Often seen in vector DB docs: `cosine distance = 1 - cosine similarity`

- Close to `0` → highly similar
- Larger value → less similar

---

## Step 6 — Build Final Prompt and Send to Claude

Combine the user question with the retrieved chunk:
```python
prompt = f"""
Answer the user's question about the report.

<user_question>
{user_question}
</user_question>

<report>
{relevant_chunk}
</report>
"""
```

Claude receives only the **relevant section** — not the entire document.

---

## Why This Works

The medical chunk contained the word "bug" and the software chunk contained
"infection vectors" — keyword search would have confused them.
Semantic search correctly identified the **meaning** of the question
and returned the right chunk despite the misleading vocabulary.

---

## Key Takeaway

> RAG works because **meaning is preserved in embedding space**.  
> Similar meaning → similar vectors → high cosine similarity.  
> This lets you retrieve the right context even when the exact words don't match.

# Implementing the RAG Pipeline

---

## The Five Steps
```
1. Chunk the text
        |
        v
2. Generate embeddings for each chunk
        |
        v
3. Store embeddings + original text in vector store
        |
        v
4. Embed the user's query
        |
        v
5. Search the store → return most similar chunks
```

---

## Step 1 — Chunk the Document
```python
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)
```

Uses `chunk_by_section()` to split on document structure (headers/sections).

---

## Step 2 — Generate Embeddings for All Chunks
```python
embeddings = generate_embedding(chunks)   # accepts a list for batch processing
```

The updated `generate_embedding()` handles both a single string and a list of strings.
Batch processing all chunks at once is more efficient than calling one at a time.

---

## Step 3 — Store in Vector Database
```python
store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})
```

**Critical:** Always store the **original text alongside the embedding**.

Why? When the search returns results, you need the actual text to include in your prompt —
not just the raw numbers.
```
Stored per entry:
    embedding  →  [0.23, -0.87, 0.41, ...]   (for searching)
    metadata   →  {"content": "chunk text"}   (for prompt building)
```

---

## Step 4 — Embed the User Query
```python
user_embedding = generate_embedding("What did the software engineering dept do last year?")
```

Use the same model and embedding function as the chunks — consistency is required
for cosine similarity to work correctly.

---

## Step 5 — Search for Relevant Chunks
```python
results = store.search(user_embedding, 2)   # return top 2 matches

for doc, distance in results:
    print(distance, "\n", doc["content"][0:200], "\n")
```

Returns `(doc, distance)` pairs sorted by cosine distance (lower = more similar).

### Example Results

| Chunk | Cosine Distance | Interpretation |
|---|---|---|
| Section 2: Software Engineering | 0.71 | Closest match |
| Methodology section | 0.72 | Second closest |

---

## Building the Final Prompt
```python
relevant_chunks = [doc["content"] for doc, distance in results]

prompt = f"""
Answer the user's question using the report sections below.

<user_question>
{user_question}
</user_question>

<report>
{"---".join(relevant_chunks)}
</report>
"""
```

---

## Key Takeaway

> RAG implementation is: **text → numbers → store → compare → retrieve → prompt**.  
> The most important detail is storing original text with each embedding —
> without it, search results are useless for building a prompt.

# BM25 Lexical Search

---

## The Problem with Semantic Search Alone

Semantic search understands **meaning** but can miss **exact term matches**.

**Example:** Searching for incident ID `INC-2023-Q4-011`

- Semantic search may return sections that are *conceptually related* but don't
  actually contain the specific ID
- A financial analysis section might score high due to context — even with no mention
  of the exact incident

---

## The Solution: Hybrid Search

Run **both** search types in parallel and merge results:
```
User Query
    |
    +------------------+------------------+
    |                                     |
    v                                     v
Semantic Search                    Lexical Search (BM25)
(embeddings, meaning)              (exact term matching)
    |                                     |
    +------------------+------------------+
                       |
                       v
               Merged Results
```

| Search Type | Strength |
|---|---|
| Semantic | Finds conceptually related content |
| Lexical (BM25) | Finds exact term and phrase matches |

---

## How BM25 Works

**BM25 (Best Match 25)** — a classic information retrieval algorithm.

### Step by Step
```
Query: "What happened with INC-2023-Q4-011?"
         |
         v
Step 1: Tokenize → ["What", "happened", "with", "INC-2023-Q4-011"]
         |
         v
Step 2: Count term frequency across all chunks
        "what"          → appears 50 times  (very common)
        "INC-2023-Q4-011" → appears 1 time  (very rare)
         |
         v
Step 3: Weight by rarity
        Common terms  → low importance
        Rare terms    → high importance
         |
         v
Step 4: Return chunks with most high-weight term matches
```

---

## Implementation
```python
# 1. Chunk the document
chunks = chunk_by_section(text)

# 2. Create BM25 store and add chunks
store = BM25Index()
for chunk in chunks:
    store.add_document({"content": chunk})

# 3. Search
results = store.search("What happened with INC-2023-Q4-011?", 3)

for doc, distance in results:
    print(distance, "\n", doc["content"][:200], "\n----\n")
```

BM25 will correctly prioritise chunks that **actually contain** `INC-2023-Q4-011`
over chunks that are merely related in topic.

---

## When BM25 Outperforms Semantic Search

| Use Case | Better approach |
|---|---|
| Incident IDs, ticket numbers, codes | BM25 |
| Technical terms, version numbers | BM25 |
| Conceptual / meaning-based questions | Semantic |
| "What does X relate to?" | Semantic |
| Exact phrase lookup | BM25 |

---

## Key Takeaway

> Semantic search and BM25 have **complementary strengths**.  
> BM25 gives high weight to rare, specific terms — exactly what semantic search
> tends to miss.  
> Combining both into a hybrid search pipeline gives you robust retrieval
> for both conceptual queries and exact term lookups.

# Multi-Index Hybrid Search — Reciprocal Rank Fusion

---

## The Architecture

Both `VectorIndex` and `BM25Index` share the same API:

| Method | VectorIndex | BM25Index |
|---|---|---|
| `add_document(doc)` | Adds chunk + embedding | Adds chunk for lexical indexing |
| `search(query, k)` | Returns chunks + cosine distances | Returns chunks + BM25 scores |

This consistency makes wrapping them in a single `Retriever` class straightforward.

---

## The Problem with Merging Raw Scores

Each index uses a **different scoring system** — you cannot directly compare or
average scores from VectorIndex and BM25Index. You need a normalised merging method.

**Solution: Reciprocal Rank Fusion (RRF)**

---

## Reciprocal Rank Fusion (RRF)

Combines results based on **rank position**, not raw scores.
Rank position is comparable across any two systems.

### Formula
```
RRF_score(d) = Σ  1 / (k + rank_i(d))
```

- `k` = smoothing constant (commonly 60; use 1 for clearer examples)
- `rank_i(d)` = rank of document `d` in the i-th index
- **Higher RRF score = more relevant chunk**

---

## Worked Example

**Query:** "What happened with INC-2023-Q4-011?"

| Text Chunk | Rank (Vector) | Rank (BM25) |
|---|---|---|
| Section 2: Software Engineering | 1 | 2 |
| Section 6: Product Engineering | 3 | 1 |
| Section 7: Historical Research | 2 | 3 |

**RRF calculation (k=1):**
```
Section 2:  1/(1+1) + 1/(1+2) = 0.500 + 0.333 = 0.833  → Rank 1
Section 6:  1/(1+3) + 1/(1+1) = 0.250 + 0.500 = 0.750  → Rank 2
Section 7:  1/(1+2) + 1/(1+3) = 0.333 + 0.250 = 0.583  → Rank 3
```

Section 2 ranked highly in **both** indexes → rises to top. Makes intuitive sense.

---

## `Retriever` Class
```python
class Retriever:
    def __init__(self, *indexes: SearchIndex):
        if len(indexes) == 0:
            raise ValueError("At least one index must be provided")
        self._indexes = list(indexes)

    def add_document(self, document: Dict[str, Any]):
        for index in self._indexes:
            index.add_document(document)       # add to all indexes

    def search(self, query_text: str, k: int = 1, k_rrf: int = 60):
        all_results = [index.search(query_text, k) for index in self._indexes]

        # Build rank table across indexes
        # Apply RRF formula to each document
        # Sort by RRF score descending
        # Return top-k merged results
```

### Usage
```python
vector_index = VectorIndex()
bm25_index   = BM25Index()

retriever = Retriever(vector_index, bm25_index)

for chunk in chunks:
    retriever.add_document({"content": chunk})   # adds to both indexes

results = retriever.search("What happened with INC-2023-Q4-011?", k=3)
```

---

## Before vs After Hybrid Search

**Query:** "What happened with INC-2023-Q4-011?"

| Rank | Vector only | Hybrid (Vector + BM25) |
|---|---|---|
| 1 | Cybersecurity (correct) | Cybersecurity (correct) |
| 2 | Financial Analysis (wrong) | Software Engineering (correct) |
| 3 | — | Legal Developments |

Hybrid search surfaces the correct sections that vector search missed.

---

## Extensibility

The `Retriever` works with **any** index that implements `add_document()` + `search()`.
Adding new search methods requires no changes to the core pipeline:
```python
# Add any new index type — Retriever handles it automatically
retriever = Retriever(vector_index, bm25_index, keyword_index, graph_index)
```

---

## Key Takeaway

> RRF solves the incompatible scores problem by comparing **ranks, not scores**.  
> A chunk that ranks highly in both indexes gets a higher combined RRF score,
> ensuring consistently good results across both semantic and lexical queries.